# Build a Small LLM from Scratch: MiniGPT Deep Dive
### AI Engineering Assignment Handout Implementation

This notebook provides a complete, self-contained, reproducible implementation of a **decoder-only causal Transformer language model (MiniGPT)** trained from scratch on the **Tiny Shakespeare** dataset without loading any pretrained weights.

**Key Highlights Covered:**
1. Deterministic character-level tokenization
2. Step-by-step numerical hand calculation of self-attention
3. Multi-Head Attention with strict upper-triangular causal masking
4. Sinusoidal and learned positional encodings
5. Pre-LayerNorm Transformer blocks with residual connections and GELU FFN
6. Target-shifted cross-entropy training pipeline
7. Autoregressive text generation with temperature and Top-K sampling
8. Multi-head attention map visualization

In [ ]:
# 1. Environment Setup & Dependency Installation
import os
import sys
import math
import time
import random
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using Device: {device}')

## 2. Dataset Acquisition & Deterministic Tokenizer
We fetch the canonical **Tiny Shakespeare** dataset (~1.11MB, 40,000 lines) and construct a deterministic character-level tokenizer.

In [ ]:
# Download Tiny Shakespeare if not present locally
DATA_PATH = 'data/input.txt'
if not os.path.exists(DATA_PATH):
    os.makedirs('data', exist_ok=True)
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    print('Downloading Tiny Shakespeare dataset...')
    urllib.request.urlretrieve(url, DATA_PATH)

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f'Corpus length: {len(raw_text):,} characters')

# Deterministic Character Tokenizer
chars = sorted(list(set(raw_text)))
vocab_size = len(chars)
char_to_id = {ch: i for i, ch in enumerate(chars)}
id_to_char = {i: ch for i, ch in enumerate(chars)}

def encode(s: str):
    return [char_to_id[c] for c in s if c in char_to_id]

def decode(l: list):
    return ''.join([id_to_char[i] for i in l])

print(f'Vocabulary size: {vocab_size} unique characters')
sample_encoded = encode('Hello Shakespeare!')
print(f'"Hello Shakespeare!" -> {sample_encoded} -> "{decode(sample_encoded)}"')

## 3. Hand Calculation of Causal Self-Attention (Handout Section 6)
Before relying on PyTorch, we compute the core attention mechanism explicitly for a 3-token sequence with $d_k=2$ and $d_v=2$:
$$\text{Scores} = \frac{Q K^T}{\sqrt{d_k}}$$
$$\text{Masked Weights} = \mathrm{softmax}(\text{Scores} + M)$$
$$\text{Output} = \text{Masked Weights} \times V$$

In [ ]:
# Explicit 3-token sequence matrices
Q = np.array([[1.0, 0.0], [0.0, 2.0], [1.0, 1.0]], dtype=float)
K = np.array([[1.0, 1.0], [0.0, 1.0], [2.0, 0.0]], dtype=float)
V = np.array([[1.0, 2.0], [3.0, 0.0], [0.0, 4.0]], dtype=float)

d_k = 2
sqrt_dk = math.sqrt(d_k)

# 1. Raw scores: Q @ K^T
raw_scores = Q @ K.T

# 2. Scaled scores
scaled_scores = raw_scores / sqrt_dk

# 3. Causal mask (lower triangular)
causal_mask = np.tril(np.ones((3, 3)))
masked_scores = np.where(causal_mask == 1, scaled_scores, -1e9)

# 4. Row-wise Softmax
exp_scores = np.exp(masked_scores - np.max(masked_scores, axis=-1, keepdims=True))
attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)

# 5. Output = weights @ V
attention_output = attention_weights @ V

print('Scaled Scores:\n', np.round(scaled_scores, 3))
print('\nCausal Attention Weights (Softmax):\n', np.round(attention_weights, 3))
print('\nWeighted Output (Weights @ V):\n', np.round(attention_output, 3))

## 4. Transformer Architecture from Scratch
We implement:
1. **Sinusoidal Positional Encoding**
2. **Multi-Head Self-Attention** with causal masking
3. **Position-wise Feed-Forward Network (GELU)**
4. **Transformer Decoder Block (Pre-LayerNorm & Residuals)**
5. **MiniGPT Model**

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        return self.dropout(x)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_context_length: int = 1024, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_context_length, max_context_length))
        self.register_buffer('causal_mask', mask.view(1, 1, max_context_length, max_context_length))

    def forward(self, x: torch.Tensor, return_attention: bool = False):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float('-inf'))
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        out = torch.matmul(weights, v)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        return (out, weights) if return_attention else (out, None)

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, max_context_length: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.ln_1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, max_context_length, dropout)
        self.ln_2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim, bias=False),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, d_model, bias=False),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor, return_attention: bool = False):
        norm_x1 = self.ln_1(x)
        attn_out, weights = self.attn(norm_x1, return_attention=return_attention)
        x = x + attn_out
        x = x + self.ffn(self.ln_2(x))
        return x, weights

class MiniGPT(nn.Module):
    def __init__(self, vocab_size: int, context_length: int = 128, d_model: int = 128, n_heads: int = 4, n_layers: int = 4, dropout: float = 0.1):
        super().__init__()
        self.context_length = context_length
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len=context_length, dropout=dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, 4 * d_model, max_context_length=context_length, dropout=dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor = None, return_attentions: bool = False):
        B, T = idx.size()
        x = self.token_embedding(idx)
        x = self.pos_encoding(x)
        all_attns = [] if return_attentions else None
        for block in self.blocks:
            x, attn = block(x, return_attention=return_attentions)
            if return_attentions:
                all_attns.append(attn)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss, all_attns

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 0.8, top_k: int = 40):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.context_length:]
            logits, _, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_token), dim=1)
        return idx

## 5. Model Initialization & Parameter Count

In [ ]:
model = MiniGPT(vocab_size=vocab_size, context_length=128, d_model=128, n_heads=4, n_layers=4).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'MiniGPT Total Parameters: {total_params:,} ({total_params/1e6:.2f}M)')

## 6. Training Pipeline & Loss Tracking

In [ ]:
# Prepare train/val datasets
data_tensor = torch.tensor(encode(raw_text), dtype=torch.long)
n_split = int(0.9 * len(data_tensor))
train_data, val_data = data_tensor[:n_split], data_tensor[n_split:]

def get_batch(split, batch_size=64, context_length=128):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(d) - context_length - 1, (batch_size,))
    x = torch.stack([d[i:i+context_length] for i in ix])
    y = torch.stack([d[i+1:i+context_length+1] for i in ix])
    return x.to(device), y.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

# Run training loop
steps, train_losses, val_losses = [], [], []
print('Starting training for 800 iterations...')
model.train()
for step in range(801):
    x, y = get_batch('train')
    logits, loss, _ = model(x, y)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    if step % 100 == 0:
        model.eval()
        with torch.no_grad():
            vx, vy = get_batch('val')
            _, vloss, _ = model(vx, vy)
        steps.append(step)
        train_losses.append(loss.item())
        val_losses.append(vloss.item())
        print(f'Step {step:4d} | Train Loss: {loss.item():.4f} | Val Loss: {vloss.item():.4f}')
        model.train()

# Plot training curve
plt.figure(figsize=(8, 4))
plt.plot(steps, train_losses, label='Train Loss', color='blue')
plt.plot(steps, val_losses, label='Val Loss', color='orange', linestyle='--')
plt.xlabel('Steps')
plt.ylabel('Cross-Entropy Loss')
plt.title('MiniGPT Training vs. Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Autoregressive Text Generation
Generating coherent text from prompts without pretrained checkpoints.

In [ ]:
prompts = [
    'KING LEAR:\n',
    'ROMEO:\n',
    'To be, or not to be, that is the '
]

for p in prompts:
    p_tensor = torch.tensor([encode(p)], dtype=torch.long, device=device)
    out = model.generate(p_tensor, max_new_tokens=120, temperature=0.8, top_k=40)[0].tolist()
    print('=' * 60)
    print(f'Prompt: {p}')
    print(f'Generated:\n{decode(out)}')
    print('=' * 60)

## 8. Multi-Head Attention Heatmap Inspection
Inspect what each attention head learns.

In [ ]:
sample_input = 'ROMEO:\nWhat light'
tokens = encode(sample_input)
x = torch.tensor([tokens], dtype=torch.long, device=device)

with torch.no_grad():
    _, _, all_attns = model(x, return_attentions=True)

last_layer_attn = all_attns[-1][0].cpu().numpy() # (n_heads, T, T)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
labels = list(sample_input)

for h in range(4):
    im = axes[h].imshow(last_layer_attn[h], cmap='magma', vmin=0, vmax=1)
    axes[h].set_title(f'Head {h+1}')
    axes[h].set_xticks(range(len(labels)))
    axes[h].set_yticks(range(len(labels)))
    axes[h].set_xticklabels(labels, fontsize=8)
    axes[h].set_yticklabels(labels, fontsize=8)

plt.suptitle('Attention Weights across Heads (Layer 4)', fontsize=14)
plt.tight_layout()
plt.show()